# Olist E-Commerce Analysis

## Exploratory Data Analysis

SQL-backed exploration of sales, delivery, customer reviews, payments, and sellers.
All business analyses use `cleaned_orders`: delivered orders with a recorded
actual delivery date. The first checks inspect the original imported tables;
the scope-filter cell then restricts payments, items, and reviews for analysis.

**Definitions:** payment value is customer payments, not profit. Item Revenue
includes price plus freight. Review averages and negative-review rates are per
review record, with scores 1 and 2 classified as negative. Category and seller
joins use distinct order/category or order/seller pairs to avoid item weighting.

Saved outputs have been cleared after the corrections. Run all cells in order
against your local MySQL database to regenerate tables and charts.

## 1. Import Libraries

Import the Python libraries required for data loading, manipulation, analysis, visualization, and MySQL connectivity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

## 2. Connect to MySQL

The analysis uses MySQL as the main data source.

The database contains the original Olist tables and the `cleaned_orders` analytical view created during the SQL phase.

In [ ]:
from getpass import getpass
from sqlalchemy import create_engine, URL

engine = create_engine(
    URL.create(
        drivername="mysql+pymysql",
        username="root",
        password=getpass("Enter your MySQL password: "),
        host="localhost",
        port=3306,
        database="olist_project",
    )
)

Test the connection

In [ ]:
pd.read_sql(
    "SELECT 1",
    con=engine
)

## 3. Load Data

The datasets are loaded directly from MySQL rather than re-importing the original CSV files.

This keeps MySQL as the central data source for the project and allows Python to perform EDA on the same data used for the SQL analysis.

In [ ]:
cleaned_orders = pd.read_sql(
    "SELECT * FROM cleaned_orders",
    con=engine
)

orders = pd.read_sql(
    "SELECT * FROM orders",
    con=engine
)

order_items = pd.read_sql(
    "SELECT * FROM order_items",
    con=engine
)

payments = pd.read_sql(
    "SELECT * FROM payments",
    con=engine
)

reviews = pd.read_sql(
    "SELECT * FROM reviews",
    con=engine
)

customers = pd.read_sql(
    "SELECT * FROM customers",
    con=engine
)

products = pd.read_sql(
    "SELECT * FROM products",
    con=engine
)

sellers = pd.read_sql(
    "SELECT * FROM sellers",
    con=engine
)

category_translation = pd.read_sql(
    "SELECT * FROM category_translation",
    con=engine
)

## 4. Dataset Overview

Before performing analysis, inspect the size and structure of each dataset.

In [ ]:
tables = {
    "cleaned_orders": cleaned_orders,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "customers": customers,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

## 5. Data Types

Check the data types of the columns to ensure that dates, numeric values, and categorical fields are stored appropriately.

In [ ]:
for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print("=" * 60)
    df.info()

## 6. Missing Value Analysis

Check for missing values before performing analysis.

Missing values may affect calculations, joins, and visualizations.

### Observation

Missing values are concentrated in several fields such as product descriptions
and review comments. These fields are not required for the current business
analysis, so they will not be used in the main calculations.

In [ ]:
missing_values = {}

for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    
    if len(missing) > 0:
        missing_values[name] = missing

missing_values

## 7. Duplicate Check

Check for duplicate records in the main analytical datasets.

In [ ]:
print(
    "Duplicate order IDs in cleaned_orders:",
    cleaned_orders["order_id"].duplicated().sum()
)

print(
    "Duplicate order IDs in orders:",
    orders["order_id"].duplicated().sum()
)

## Apply the delivered-order scope
Keep the raw tables above for data-quality checks. All analysis below uses only order IDs in `cleaned_orders`.


In [ ]:
# cleaned_orders is the shared population for the SQL, notebook, and dashboard.
assert cleaned_orders["order_id"].notna().all()
assert cleaned_orders["order_id"].is_unique
eligible_order_ids = set(cleaned_orders["order_id"])
payments = payments.loc[payments["order_id"].isin(eligible_order_ids)].copy()
order_items = order_items.loc[order_items["order_id"].isin(eligible_order_ids)].copy()
reviews = reviews.loc[reviews["order_id"].isin(eligible_order_ids)].copy()
assert reviews["review_score"].between(1, 5).all(), "Review scores need investigation."
print("Eligible delivered orders:", len(eligible_order_ids))
print("Scoped payment records:", len(payments))
print("Scoped item rows:", len(order_items))
print("Scoped review records:", len(reviews))


# 8. Sales Exploratory Data Analysis

This section explores marketplace revenue, order volume, product category performance, and average order value.

### Key questions

1. How has marketplace revenue changed over time?
2. Which product categories generate the highest revenue?
3. What is the average customer spending per order?
4. Is revenue growth driven by more orders or higher spending?

## 8.1 Monthly Revenue Trend

Calculate total revenue by month using payment values.

Payment records are first aggregated at the order level because a single order can contain multiple payment records.

In [ ]:
order_revenue = (
    payments.groupby("order_id", as_index=False)["payment_value"].sum()
)

# Keep all eligible delivered orders, including any missing payment record.
monthly_revenue = cleaned_orders[
    ["order_id", "order_purchase_timestamp"]
].merge(order_revenue, on="order_id", how="left", validate="one_to_one")
monthly_revenue["payment_value"] = monthly_revenue["payment_value"].fillna(0)
monthly_revenue["month"] = pd.to_datetime(
    monthly_revenue["order_purchase_timestamp"]
).dt.to_period("M")
revenue = monthly_revenue.groupby("month")["payment_value"].sum()

In [ ]:
plt.figure(figsize=(12, 5))

revenue.plot(
    kind="line",
    title="Monthly Revenue"
)

plt.ylabel("Revenue")
plt.xlabel("Month")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 8.2 Monthly Order Volume

Analyze the number of unique orders placed each month.

Comparing order volume with revenue helps determine whether revenue changes are driven by changes in the number of orders.

In [ ]:
monthly_orders = (
    monthly_revenue
    .groupby("month")["order_id"]
    .nunique()
)

In [ ]:
plt.figure(figsize=(12, 5))

monthly_orders.plot(
    kind="line",
    title="Monthly Order Volume"
)

plt.ylabel("Number of Orders")
plt.xlabel("Month")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 8.3 Monthly Average Order Value

Average Order Value (AOV) measures the average amount spent per order.

A monthly AOV analysis helps determine whether changes in revenue are driven primarily by changes in order volume or by changes in the average amount spent per order.

AOV is calculated as:

**AOV = Total Revenue / Number of Unique Orders**

This provides additional context when interpreting monthly revenue and order-volume trends.


In [ ]:
average_order_value = (
    payments["payment_value"].sum() / cleaned_orders["order_id"].nunique()
)
print(f"Average Order Value: {average_order_value:.2f}")

In [ ]:
monthly_aov = (
    monthly_revenue
    .groupby("month")
    .agg(
        total_revenue=("payment_value", "sum"),
        total_orders=("order_id", "nunique")
    )
)

monthly_aov["aov"] = (
    monthly_aov["total_revenue"]
    / monthly_aov["total_orders"]
)

monthly_aov.round(2)

## 8.4 Product Category Revenue

Analyze which product categories generate the highest revenue.

For this analysis, order-item revenue is defined as:

Order Item Value = Product Price + Freight Value

This represents the gross value associated with each order item,
including the shipping charge.

In [ ]:
category_revenue = (
    cleaned_orders[
        ["order_id"]
    ]
    .merge(
        order_items[
            [
                "order_id",
                "product_id",
                "price",
                "freight_value"
            ]
        ],
        on="order_id",
        how="inner"
    )
    .merge(
        products[
            [
                "product_id",
                "product_category_name"
            ]
        ],
        on="product_id",
        how="inner"
    )
    .merge(
        category_translation[
            [
                "product_category_name",
                "product_category_name_english"
            ]
        ],
        on="product_category_name",
        how="left"
    )
)

Create revenue:

In [ ]:
category_revenue["item_revenue"] = (
    category_revenue["price"]
    + category_revenue["freight_value"]
)

Aggregate:

In [ ]:
category_summary = (
    category_revenue
    .groupby("product_category_name_english")
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("item_revenue", "sum")
    )
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

category_summary.head(10)

### Top 10 Product Categories by Revenue

In [ ]:
top_categories = (
    category_summary["total_revenue"]
    .head(10)
    .sort_values()
)

In [ ]:
plt.figure(figsize=(10, 6))

top_categories.plot(
    kind="barh",
    title="Top 10 Product Categories by Revenue"
)

plt.xlabel("Revenue")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()

# 9. Delivery Exploratory Data Analysis

This section explores delivery performance and investigates whether delivery performance is associated with customer satisfaction.

### Key questions

1. How many orders were delivered late?
2. How long do deliveries take?
3. Which states have the highest late-delivery rates?
4. Are there unusual delivery times or outliers?
5. Is delivery performance associated with customer review scores?

## 9.1 Delivery Status Distribution

Analyze the number and percentage of orders that were delivered on time versus late.

In [ ]:
delivery_summary = (
    cleaned_orders
    .groupby("delivery_status")["order_id"]
    .count()
)

delivery_percentage = (
    delivery_summary
    / delivery_summary.sum()
    * 100
)

delivery_summary

In [ ]:
delivery_percentage.round(2)

In [ ]:
plt.figure(figsize=(8, 5))

delivery_summary.plot(
    kind="bar",
    title="Delivery Performance"
)

plt.ylabel("Number of Orders")
plt.xlabel("Delivery Status")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 9.2 Average Delivery Time

Compare actual delivery time and delivery delay between late and on-time orders.

In [ ]:
delivery_time_summary = (
    cleaned_orders
    .groupby("delivery_status")
    .agg(
        average_delivery_days=("delivery_days", "mean"),
        average_delay_days=("delay_days", "mean")
    )
    .round(2)
)

delivery_time_summary

In [ ]:
plt.figure(figsize=(8, 5))

delivery_time_summary[
    "average_delivery_days"
].plot(
    kind="bar",
    title="Average Delivery Time by Delivery Status"
)

plt.ylabel("Average Delivery Days")
plt.xlabel("Delivery Status")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 9.3 Late Delivery Rate by State

Analyze whether delivery performance varies across customer states.

The late-delivery rate is calculated as:

Late Delivery Rate = Late Orders / Total Orders × 100

In [ ]:
delivery_state = (
    cleaned_orders[
        [
            "order_id",
            "customer_id",
            "delivery_status"
        ]
    ]
    .merge(
        customers[
            [
                "customer_id",
                "customer_state"
            ]
        ],
        on="customer_id",
        how="inner"
    )
)

delivery_state["is_late"] = (
    delivery_state["delivery_status"] == "Late"
)

In [ ]:
state_delivery = (
    delivery_state
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "nunique"),
        late_orders=("is_late", "sum")
    )
)

state_delivery["late_delivery_rate"] = (
    state_delivery["late_orders"]
    / state_delivery["total_orders"]
    * 100
)

state_delivery = state_delivery.sort_values(
    "late_delivery_rate",
    ascending=False
)

state_delivery.head(10)

In [ ]:
state_delivery_filtered = state_delivery[
    state_delivery["total_orders"] >= 100
]

top_late_states = (
    state_delivery_filtered
    .sort_values("late_delivery_rate", ascending=False)
    .head(10)
)

top_late_states

### Top 10 States by Late Delivery Rate

In [ ]:
top_late_states_chart = (
    state_delivery_filtered["late_delivery_rate"]
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10, 6))

top_late_states_chart.plot(
    kind="barh",
    title="Top 10 States by Late Delivery Rate (min. 100 orders)"
)

plt.xlabel("Late Delivery Rate (%)")
plt.ylabel("Customer State")

plt.tight_layout()
plt.show()

In [ ]:
# Show only the late-rate series, rather than mixing counts and percentages.
ax = top_late_states["late_delivery_rate"].sort_values().plot(
    kind="barh", figsize=(10, 6),
    title="Top 10 States by Late Delivery Rate (min. 100 orders)"
)
ax.set_xlabel("Late Delivery Rate (%)")
ax.set_ylabel("Customer State")
plt.tight_layout()
plt.show()

## 9.4 Delivery Time Distribution

Examine the distribution of delivery times to identify common delivery durations and potential outliers.

In [ ]:
plt.figure(figsize=(10, 5))

cleaned_orders["delivery_days"].plot(
    kind="hist",
    bins=30,
    title="Distribution of Delivery Time"
)

plt.xlabel("Delivery Days")
plt.ylabel("Number of Orders")

plt.tight_layout()
plt.show()

In [ ]:
cleaned_orders["delivery_days"].describe()

## 9.5 Delivery Performance vs Customer Satisfaction

Investigate whether late delivery is associated with lower customer review scores.

In [ ]:
delivery_review = (
    cleaned_orders[
        [
            "order_id",
            "delivery_days",
            "delay_days",
            "delivery_status"
        ]
    ]
    .merge(
        reviews[
            [
                "order_id",
                "review_score"
            ]
        ],
        on="order_id",
        how="inner"
    )
)

Calculate:

In [ ]:
review_by_delivery = (
    delivery_review
    .groupby("delivery_status")["review_score"]
    .mean()
    .round(2)
)

review_by_delivery

In [ ]:
plt.figure(figsize=(8, 5))

review_by_delivery.plot(
    kind="bar",
    title="Average Review Score by Delivery Status"
)

plt.ylabel("Average Review Score")
plt.xlabel("Delivery Status")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 9.6 Delivery Time vs Review Score

Explore whether longer delivery times are associated with lower customer review scores.

A scatter plot is used to examine the relationship between delivery duration and review score.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    delivery_review["delivery_days"],
    delivery_review["review_score"],
    alpha=0.2
)

plt.title("Delivery Time vs Review Score")
plt.xlabel("Delivery Days")
plt.ylabel("Review Score")

plt.tight_layout()
plt.show()

Correlation:

In [ ]:
delivery_review[
    [
        "delivery_days",
        "review_score"
    ]
].corr()

# 10. Payment Exploratory Data Analysis

This section examines customer payment behavior, including payment method usage and payment value.

In [ ]:
payment_summary = (
    payments
    .groupby("payment_type")
    .agg(
        transactions=("order_id", "count"),
        total_revenue=("payment_value", "sum")
    )
    .sort_values(
        "transactions",
        ascending=False
    )
)

payment_summary

## 10.1 Payment Method Usage

Analyze which payment methods are most frequently used by customers.

In [ ]:
payment_summary

In [ ]:
plt.figure(figsize=(8, 5))

payment_summary["transactions"].plot(
    kind="bar",
    title="Payment Method Usage"
)

plt.xlabel("Payment Method")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 10.2 Revenue by Payment Method

Compare the total revenue associated with each payment method.

In [ ]:
payment_revenue = (
    payments
    .groupby("payment_type")["payment_value"]
    .sum()
    .sort_values(ascending=False)
)

payment_revenue

In [ ]:
plt.figure(figsize=(8, 5))

payment_revenue.plot(
    kind="bar",
    title="Revenue by Payment Method"
)

plt.xlabel("Payment Method")
plt.ylabel("Revenue")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 10.3 Installment Payment Behavior

The chart compares payment-record averages. Zero or missing installment counts
are classified as Unspecified. The separate order-level summary classifies each
order once, matching the dashboard's installment card and donut.

In [ ]:
payments["payment_group"] = np.select(
    [payments["payment_installments"] > 1,
     payments["payment_installments"] == 1],
    ["Installment", "Single Payment"],
    default="Unspecified"
)

order_payment_plan = payments.groupby("order_id")["payment_installments"].max()
order_payment_plan = order_payment_plan.gt(1).map(
    {True: "Uses installments", False: "No installments"}
)
order_payment_summary = order_payment_plan.value_counts().rename("payment_orders").to_frame()
order_payment_summary["percentage"] = (
    100 * order_payment_summary["payment_orders"] / order_payment_summary["payment_orders"].sum()
)
display(order_payment_summary.round(2))

In [ ]:
installment_summary = (
    payments
    .groupby("payment_group")
    .agg(
        transactions=("order_id", "count"),
        average_payment_value=("payment_value", "mean")
    )
    .round(2)
)

installment_summary

In [ ]:
plt.figure(figsize=(8, 5))

installment_summary["average_payment_value"].plot(
    kind="bar",
    title="Average Payment Value: Single vs Installment"
)

plt.xlabel("Payment Group")
plt.ylabel("Average Payment Value")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

# 11. Customer Satisfaction Exploratory Data Analysis

This section explores customer satisfaction using review scores and investigates
whether delivery performance and product categories are associated with customer
satisfaction.

### Key questions

1. What is the overall average customer review score?
2. How are review scores distributed?
3. Does late delivery affect customer satisfaction?
4. Which product categories receive the most negative reviews?
5. Are there categories with high revenue but relatively low satisfaction?

## 11.1 Overall Customer Satisfaction

Calculate the mean of review records belonging to eligible delivered orders.
An order may have more than one review record; no item-row weighting is used.

In [ ]:
average_review_score = reviews["review_score"].mean()

print(f"Average Customer Review Score: {average_review_score:.2f}")

In [ ]:
review_summary = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

review_summary

## 11.2 Review Score Distribution

Examine the distribution of delivered-order review records across scores 1 to 5.

In [ ]:
plt.figure(figsize=(8, 5))

review_summary.plot(
    kind="bar",
    title="Customer Review Score Distribution"
)

plt.xlabel("Review Score")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 11.3 Negative Reviews by Product Category

Negative = score 1 or 2. The denominator is review records, matching Power BI.
Each review appears once per category in its order, regardless of item quantity.
The full table includes small categories; the EDA ranking requires 20 reviews.
That sample threshold can make this ranking differ from an unfiltered PBI Top 5.

In [ ]:
order_category = (
    order_items[["order_id", "product_id"]]
    .merge(products[["product_id", "product_category_name"]],
           on="product_id", how="left", validate="many_to_one")
    .merge(category_translation[["product_category_name", "product_category_name_english"]],
           on="product_category_name", how="left", validate="many_to_one")
    [["order_id", "product_category_name_english"]]
    .drop_duplicates()
)
# Keep review records intact. Deduplicate only the order/category mapping.
negative_reviews = order_category.merge(
    reviews[["order_id", "review_score"]], on="order_id", how="inner"
)

Create the negative-review flag:

In [ ]:
negative_reviews["is_negative"] = negative_reviews["review_score"].isin([1, 2])

Aggregate:

In [ ]:
category_review_summary = (
    negative_reviews.groupby("product_category_name_english", dropna=False)
    .agg(reviewed_orders=("order_id", "nunique"),
         total_reviews=("review_score", "count"),
         negative_reviews=("is_negative", "sum"))
)
category_review_summary["negative_review_rate"] = (
    100 * category_review_summary["negative_reviews"] / category_review_summary["total_reviews"]
)
# Omit unknown categories from the ranked chart, but retain them in the table.
category_review_summary_filtered = category_review_summary[
    (category_review_summary["total_reviews"] >= 20)
    & category_review_summary.index.notna()
].sort_values("negative_review_rate", ascending=False)
category_review_summary_filtered.head(10)

## 11.4 Visualize Negative Review Rates

Top 10 named categories with at least 20 review records.

In [ ]:
top_negative_categories = (
    category_review_summary_filtered[
        "negative_review_rate"
    ]
    .head(10)
    .sort_values()
)

In [ ]:
plt.figure(figsize=(10, 6))

top_negative_categories.plot(
    kind="barh",
    title="Top 10 Product Categories by Negative Review Rate"
)

plt.xlabel("Negative Review Rate (%)")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()

## 11.5 High Revenue but Low Customer Satisfaction

Identify product categories that generate substantial revenue but receive
relatively low customer review scores.

This helps identify categories that may generate strong sales while presenting
potential customer experience issues.

In [ ]:
category_reviews = order_category.merge(
    reviews[["order_id", "review_score"]], on="order_id", how="inner"
)

In [ ]:
category_satisfaction = (
    category_reviews.groupby("product_category_name_english", dropna=False)
    .agg(average_review_score=("review_score", "mean"),
         total_reviews=("review_score", "count"),
         reviewed_orders=("order_id", "nunique"))
)
category_satisfaction.round(2)

In [ ]:
category_performance = category_summary.merge(
    category_satisfaction,
    left_index=True,
    right_index=True,
    how="inner"
)

category_performance = category_performance.sort_values(
    "total_revenue",
    ascending=False
)

category_performance.head(10)

In [ ]:
top_revenue_categories = category_performance[
    category_performance["total_revenue"] >=
    category_performance["total_revenue"].quantile(0.75)
]

satisfaction_threshold = top_revenue_categories["average_review_score"].quantile(0.25)

high_revenue_low_satisfaction = (
    top_revenue_categories[
        top_revenue_categories["average_review_score"] <= satisfaction_threshold
    ]
    .sort_values("average_review_score")
)

high_revenue_low_satisfaction

# 12. Seller Performance Exploratory Data Analysis

This section evaluates seller performance based on revenue generation,
order volume, and customer satisfaction.

### Key questions

1. Which sellers generate the highest revenue?
2. Which sellers handle the highest number of orders?
3. Which sellers generate high revenue but have relatively low customer satisfaction?
4. Are there sellers that may require further investigation?

## 12.1 Top Sellers by Revenue

Identify the sellers generating the highest revenue from delivered orders.

Seller revenue is calculated using product price and freight value.

In [ ]:
seller_revenue = (
    cleaned_orders[
        ["order_id"]
    ]
    .merge(
        order_items[
            [
                "order_id",
                "seller_id",
                "price",
                "freight_value"
            ]
        ],
        on="order_id",
        how="inner"
    )
)

Calculate seller revenue:

In [ ]:
seller_revenue["seller_revenue"] = (
    seller_revenue["price"]
    + seller_revenue["freight_value"]
)

Aggregate:

In [ ]:
seller_summary = (
    seller_revenue
    .groupby("seller_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("seller_revenue", "sum")
    )
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

seller_summary.head(10)

## 12.2 Top 10 Sellers Visualization
### Top 10 Sellers by Revenue

In [ ]:
top_sellers = (
    seller_summary[
        "total_revenue"
    ]
    .head(10)
    .sort_values()
)

In [ ]:
plt.figure(figsize=(10, 6))

top_sellers.plot(
    kind="barh",
    title="Top 10 Sellers by Revenue"
)

plt.xlabel("Revenue")
plt.ylabel("Seller ID")

plt.tight_layout()
plt.show()

## 12.3 Top Sellers by Order Volume

Analyze which sellers handle the highest number of orders.

Comparing order volume with revenue helps identify whether seller performance
is driven by order quantity or higher-value products.

In [ ]:
top_sellers_orders = (
    seller_summary[
        "total_orders"
    ]
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)

In [ ]:
plt.figure(figsize=(10, 6))

top_sellers_orders.plot(
    kind="barh",
    title="Top 10 Sellers by Order Volume"
)

plt.xlabel("Number of Orders")
plt.ylabel("Seller ID")

plt.tight_layout()
plt.show()

## 12.4 Seller Customer Satisfaction

Associate order-level reviews with each seller in that order. A review is counted
once per seller, even if the seller supplied multiple items. Multi-seller order
reviews cannot establish which individual seller caused the customer's rating.

In [ ]:
order_seller = order_items[["order_id", "seller_id"]].drop_duplicates()
seller_reviews = order_seller.merge(
    reviews[["order_id", "review_score"]], on="order_id", how="inner"
)

Aggregate:

In [ ]:
seller_satisfaction = (
    seller_reviews.groupby("seller_id")
    .agg(total_reviews=("review_score", "count"),
         reviewed_orders=("order_id", "nunique"),
         average_review_score=("review_score", "mean"))
)
seller_satisfaction.round(2)

## 12.5 High Revenue but Low Customer Satisfaction

Rank sellers with at least 20 delivered orders and a review-record mean below 3.5,
matching the eligibility rule in `07_seller_analysis.sql`.

In [ ]:
seller_performance = seller_summary.merge(
    seller_satisfaction, left_index=True, right_index=True, how="left",
    validate="one_to_one"
)

Filter:

In [ ]:
high_revenue_low_satisfaction = seller_performance[
    (seller_performance["total_orders"] >= 20)
    & (seller_performance["average_review_score"] < 3.5)
].sort_values("total_revenue", ascending=False)
high_revenue_low_satisfaction.head(10)

## 12.6 Seller Revenue vs Customer Satisfaction

Show sellers with at least 20 delivered orders. Ratings are order-level reviews
associated with sellers, not independent seller-specific ratings.

In [ ]:
seller_plot = seller_performance[seller_performance["total_orders"] >= 20]
plt.figure(figsize=(10, 6))
plt.scatter(seller_plot["total_revenue"], seller_plot["average_review_score"], alpha=0.5)
plt.xlabel("Item Revenue Including Freight")
plt.ylabel("Average Review Score")
plt.title("Seller Revenue vs Customer Satisfaction (min. 20 delivered orders)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    category_performance["total_revenue"],
    category_performance["average_review_score"],
    alpha=0.5
)

plt.xlabel("Total Revenue")
plt.ylabel("Average Review Score")
plt.title("Product Category Revenue vs Customer Satisfaction")

plt.tight_layout()
plt.show()

# 13. EDA Summary and Interpretation

- All business analyses use delivered orders with an actual delivery date.
- AOV uses all eligible delivered orders; payment-method averages use payment records.
- Reviews are counted once per order/category or order/seller association, not once per item.
- Negative-review rates use review records, not the number of orders with a negative review.
- The category negative-review chart requires 20 reviews; the full table has no minimum.
- Seller comparisons use at least 20 delivered orders where stated.
- Blank categories are retained in the full review summary but excluded from named-category charts.
- Dataset boundaries contain partial periods. Compare complete periods when discussing growth.
- Delivery/review relationships describe associations, not proof of causation.

Run the notebook before drawing new conclusions or exporting charts. The README
contains the previously reconciled dashboard totals; this notebook's outputs must
be regenerated from the local MySQL database.